# Labgate 2PP Control Platform — Interactive API Testing Notebook

This notebook provides a complete, interactive walkthrough for testing the **Labgate 2PP API Platform** locally in Jupyter. It covers system discovery, single-command execution, multi-operation experiment sweeps, plan approval lifecycles, dry-run previews, telemetry retrieval, image rendering, and 3D STL model uploads.

### Prerequisites
Start the local server before running this notebook:
```bash
labgate-serve
# or via Docker: docker compose up -d
```
Server running at `http://127.0.0.1:8523`.

## 1. Environment & Helper Functions Setup

In [ ]:
import json
import os
import time
from io import BytesIO
from pathlib import Path

import requests
import yaml
from PIL import Image
import matplotlib.pyplot as plt

BASE_URL = os.environ.get("LABGATE_URL", "http://127.0.0.1:8523")


def load_tokens(tokens_file: str = "../config/tokens.yaml") -> tuple[str, str]:
    """Resolve an operator and an approver token.

    Order: environment variables first (LABGATE_OPERATOR_TOKEN /
    LABGATE_APPROVER_TOKEN), otherwise read them out of the local
    config/tokens.yaml. Real tokens are never written into this notebook,
    which is committed to git.
    """
    env_op = os.environ.get("LABGATE_OPERATOR_TOKEN")
    env_ap = os.environ.get("LABGATE_APPROVER_TOKEN")
    if env_op and env_ap:
        return env_op, env_ap

    path = Path(tokens_file)
    if not path.exists():                      # also allow running from repo root
        path = Path("config/tokens.yaml")
    if not path.exists():
        raise SystemExit(
            "No tokens found.\n"
            "  1) cp config/tokens.example.yaml config/tokens.yaml\n"
            "  2) replace each CHANGE-ME-* key with:\n"
            "       python -c \"import secrets; print(secrets.token_urlsafe(24))\"\n"
            "  3) set  labgate.tokens_file: \"config/tokens.yaml\"  in config/default.yaml\n"
            "  4) start the server with LABGATE_CONFIG=config/default.yaml"
        )

    tokens = (yaml.safe_load(path.read_text()) or {}).get("tokens") or {}
    operator = approver = None
    for token, ident in tokens.items():
        roles = set(ident.get("roles", []))
        if operator is None and {"operator", "admin"} & roles:
            operator = token
        if approver is None and {"approver", "admin"} & roles:
            approver = token
    if not operator or not approver:
        raise SystemExit("tokens.yaml needs one operator and one approver identity.")
    return operator, approver


OPERATOR_TOKEN, APPROVER_TOKEN = load_tokens()

HEADERS_OP = {"Authorization": f"Bearer {OPERATOR_TOKEN}", "Content-Type": "application/json"}
HEADERS_AP = {"Authorization": f"Bearer {APPROVER_TOKEN}", "Content-Type": "application/json"}

health = requests.get(f"{BASE_URL}/health", timeout=5).json()
print(f"Connected to {BASE_URL}  ·  mode={health['mode']}  ·  v{health['version']}")
print(f"operator token …{OPERATOR_TOKEN[-6:]}   approver token …{APPROVER_TOKEN[-6:]}")


## 2. System Health, Grounding Capabilities, & Devices
Check server liveness, machine capabilities (grounding limits), and real-time device states.

In [ ]:
# 2.1 GET /health (Open endpoint)
res = requests.get(f"{BASE_URL}/health")
print("HEALTH:", res.status_code, res.json())

# 2.2 GET /capabilities (Grounding bounds)
res = requests.get(f"{BASE_URL}/capabilities", headers=HEADERS_OP)
print("\nCAPABILITIES (abridged):")
caps = res.json()
for dev in caps.get("devices", []):
    print(f"  - Device: {dev.get('device_id')} ({dev.get('kind')})")

# 2.3 GET /devices (Live device positions & status)
res = requests.get(f"{BASE_URL}/devices", headers=HEADERS_OP)
print("\nDEVICES:", json.dumps(res.json(), indent=2))

## 3. Single Command Execution Test
Test executing individual atomic commands (`MoveStage`, `SetLaserPower`, `WriteLine`, `CaptureImage`) wrapped as single-operation plans.

In [ ]:
# 3.1 Single Command: Move Stage to (1.0, 2.0, 6.0)
single_move_spec = {
    "spec": {
        "spec_version": "1.0",
        "title": "Single Command Move Stage",
        "operations": [
            {"op": "move_stage", "target_mm": [1.0, 2.0, 6.0]}
        ]
    }
}
res = requests.post(f"{BASE_URL}/plans", json=single_move_spec, headers=HEADERS_OP)
plan_info = res.json()
print("SUBMIT SINGLE MOVE:", res.status_code, plan_info)
single_plan_id = plan_info["plan_id"]

# Approve & Execute single command
requests.post(f"{BASE_URL}/plans/{single_plan_id}/approve", headers=HEADERS_AP)
requests.post(f"{BASE_URL}/plans/{single_plan_id}/execute", headers=HEADERS_OP)
print(f"Executing single plan {single_plan_id}...")

# Poll until done
for _ in range(10):
    st = requests.get(f"{BASE_URL}/plans/{single_plan_id}", headers=HEADERS_OP).json()
    if st["state"] in ["completed", "failed", "aborted"]:
        print("SINGLE MOVE FINAL STATE:", st["state"])
        break
    time.sleep(0.2)

## 4. Full Multi-Op Experiment Sweep Lifecycle
Run a complete multi-operation experiment sweep: Submit plan $\rightarrow$ Dry-Run Preview $\rightarrow$ Human Approval $\rightarrow$ Execution $\rightarrow$ Results & Image Retrieval.

In [ ]:
# 4.1 Submit Power Sweep Array Plan
sweep_spec = {
    "spec": {
        "spec_version": "1.0",
        "title": "Power sweep 3 lines + inspection",
        "description": "Sweep 10%, 20%, 30% power over 3 parallel 4mm lines",
        "operations": [
            {
                "op": "write_power_sweep_array",
                "x_start_mm": -2.0, "x_end_mm": 2.0,
                "y_start_mm": 0.0, "y_pitch_mm": 0.1,
                "attenuator_percent_per_line": [10.0, 20.0, 30.0],
                "z_mm": 6.0, "velocity_mm_s": 5.0
            },
            {
                "op": "capture_image",
                "label": "sweep_result",
                "wl_on": True
            }
        ]
    }
}

res = requests.post(f"{BASE_URL}/plans", json=sweep_spec, headers=HEADERS_OP)
sweep_plan = res.json()
sweep_plan_id = sweep_plan["plan_id"]
print(f"Submitted Sweep Plan ID: {sweep_plan_id} (State: {sweep_plan['state']})")
print("Validation Summary:", sweep_plan.get("validation_report", {}).get("summary"))

In [ ]:
# 4.2 Dry-Run & Render Toolpath Preview
dry_run_res = requests.post(f"{BASE_URL}/plans/{sweep_plan_id}/dry-run", headers=HEADERS_OP).json()
print("DRY-RUN ESTIMATES:")
print(f"  - Total Duration: {dry_run_res.get('total_duration_s', 0):.2f} seconds")
print(f"  - Stage Traversal: {dry_run_res.get('stage_distance_mm', 0):.2f} mm")
print(f"  - Exposure Count: {dry_run_res.get('exposure_count', 0)}")

# Display Preview Artifact PNG in Jupyter
preview_artifact = dry_run_res.get("preview_artifact")
if preview_artifact:
    img_res = requests.get(f"{BASE_URL}/plans/{sweep_plan_id}/results/artifacts/{preview_artifact}", headers=HEADERS_OP)
    if img_res.status_code == 200:
        img = Image.open(BytesIO(img_res.content))
        plt.figure(figsize=(6, 6))
        plt.imshow(img)
        plt.title("Rendered Toolpath Preview")
        plt.axis("off")
        plt.show()

In [ ]:
# 4.3 Approve Plan (Using Approver Token)
app_res = requests.post(f"{BASE_URL}/plans/{sweep_plan_id}/approve", headers=HEADERS_AP)
print("APPROVAL STATUS:", app_res.status_code, app_res.json()["state"])

# 4.4 Execute Approved Plan
exe_res = requests.post(f"{BASE_URL}/plans/{sweep_plan_id}/execute", headers=HEADERS_OP)
print("EXECUTE ENQUEUED:", exe_res.status_code, exe_res.json()["state"])

# 4.5 Poll Queue & Execution Status
while True:
    st = requests.get(f"{BASE_URL}/plans/{sweep_plan_id}", headers=HEADERS_OP).json()
    current_state = st["state"]
    print(f"Polling plan state: {current_state}...")
    if current_state in ["completed", "failed", "aborted"]:
        print(f"\nPlan execution reached terminal state: {current_state}")
        break
    time.sleep(0.3)

In [ ]:
# 4.6 Fetch Results Manifest & Telemetry
res_manifest = requests.get(f"{BASE_URL}/plans/{sweep_plan_id}/results", headers=HEADERS_OP).json()
print("RESULTS MANIFEST:", json.dumps(res_manifest.get("manifest", {}), indent=2))

# Display Captured Inspection Image
img_artifact = "sweep_result.png"
img_res = requests.get(f"{BASE_URL}/plans/{sweep_plan_id}/results/artifacts/{img_artifact}", headers=HEADERS_OP)
if img_res.status_code == 200:
    img = Image.open(BytesIO(img_res.content))
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.title("Captured Inspection Photo (sweep_result.png)")
    plt.axis("off")
    plt.show()

## 5. 3D Model Upload & Slicing Test
Upload a sample 3D STL file, inspect metadata, and submit a 3D print plan.

In [ ]:
# 5.1 Upload sample STL model
from pathlib import Path
model_path = Path("labgate_data/models/cube_100.stl")
if model_path.exists():
    with open(model_path, "rb") as f:
        res = requests.post(f"{BASE_URL}/models", files={"file": f}, headers={"Authorization": f"Bearer {OPERATOR_TOKEN}"})
    model_info = res.json()
    print("STL UPLOADED:", res.status_code, model_info)
    model_id = model_info.get("model_id")
else:
    # List existing models if cube_100.stl is not locally present
    models = requests.get(f"{BASE_URL}/models", headers=HEADERS_OP).json()
    print("EXISTING MODELS:", models)
    model_id = models[0]["model_id"] if models else None

if model_id:
    print(f"\nUsing Model ID: {model_id}")
    # 5.2 Submit STL Print Plan
    stl_spec = {
        "spec": {
            "spec_version": "1.0",
            "title": "3D Print Cube Model",
            "operations": [
                {
                    "op": "print_stl",
                    "model_id": model_id,
                    "unit": "micron",
                    "step_size": 5.0,
                    "start_position_mm": [0.0, 0.0, 6.0],
                    "attenuator_percent": 25.0,
                    "velocity_mm_s": 5.0
                }
            ]
        }
    }
    stl_plan = requests.post(f"{BASE_URL}/plans", json=stl_spec, headers=HEADERS_OP).json()
    print("STL PLAN SUBMITTED:", stl_plan["plan_id"], stl_plan["state"])